# Step 3 — Merge & Normalize Extracted Skills

Combines the LLM-extracted skills checkpoint with the deduplicated vacancies parquet, then:
1. **Case-insensitive deduplication** of skill strings within each vacancy.
2. **Blacklist filter** removes non-technical noise (human languages, soft skills, seniority labels).
3. **Normalization map** folds spelling variants into canonical names.
4. **Version stripping** — `Python 3.10` → `Python`.

**Runtime:** CPU, < 30 seconds.

**Inputs:**
- `step2_master_vacancies_clustered.parquet`
- `step3_skills_checkpoint.json`

## 1. Paths

Adjust to your environment (Kaggle / Colab / local).

In [ ]:
import json
import re
from collections import Counter
import pandas as pd

INPUT_PARQUET   = 'path/to/step2_master_vacancies_clustered.parquet'
CHECKPOINT_JSON = 'path/to/step3_skills_checkpoint.json'
OUTPUT_PATH     = 'path/to/step3_vacancies_with_skills.parquet'

## 2. Load Inputs

In [6]:
with open(CHECKPOINT_JSON) as f:
    raw_skills = json.load(f)

df = pd.read_parquet(INPUT_PARQUET)
df_p = df[df['is_parent'] == True].reset_index(drop=True).copy()

print(f'Parent vacancies:   {len(df_p):,}')
print(f'Checkpoint records: {len(raw_skills):,}')
print(f'Matched:            {len(set(raw_skills) & set(df_p["global_id"])):,}')

Parent vacancies:   27,720
Checkpoint records: 27,720
Matched:            27,720


## 3. Cleaning Rules

The blacklist and normalization map were extended after inspecting the top-200 raw extracted skills.
Both the **lower-case keys** and the **lower-case input** are compared, so case variants fold automatically.

In [7]:
BLACKLIST = {
    # Human languages
    'english', 'ukrainian', 'russian', 'polish', 'german', 'spanish', 'french',
    'italian', 'czech', 'portuguese', 'mandarin', 'chinese', 'japanese',
    # Soft skills
    'communication', 'communication skills', 'teamwork', 'leadership',
    'problem solving', 'problem-solving', 'analytical', 'analytical thinking',
    'critical thinking', 'attention to detail', 'time management',
    'multitasking', 'self-motivated', 'proactive', 'adaptability',
    'collaboration', 'creativity', 'work ethic', 'organization',
    'interpersonal skills', 'presentation skills',
    # Seniority / role labels
    'developer', 'engineer', 'senior', 'junior', 'middle', 'lead',
    'manager', 'architect', 'specialist', 'consultant', 'expert',
    'principal', 'staff', 'intern', 'trainee',
    # Generic categories
    'backend', 'back-end', 'back end',
    'frontend', 'front-end', 'front end',
    'fullstack', 'full-stack', 'full stack',
    'software development', 'web development', 'mobile development',
    'programming', 'coding', 'software engineering',
    # Generic / vague terms seen in raw output
    'devops', 'analytics', 'data analysis', 'project management',
    'cloud platforms', 'monitoring', 'networking', 'optimization',
    'apis', 'nosql', 'nosql databases', 'unit testing', 'ai', 'crm',
    # Education / general
    'computer science', 'cs degree', 'mathematics', 'statistics',
    'bachelor', 'master', 'phd', 'university',
    # Work mode
    'remote', 'remote work', 'hybrid', 'office', 'on-site', 'onsite',
}

NORMALIZATION_MAP = {
    # --- Languages ---
    'js': 'JavaScript', 'java script': 'JavaScript', 'javascript es6': 'JavaScript',
    'es6': 'JavaScript', 'es2015': 'JavaScript',
    'ts': 'TypeScript', 'type script': 'TypeScript',
    'py': 'Python', 'python3': 'Python', 'python 3': 'Python', 'python 2': 'Python',
    'cpp': 'C++', 'c plus plus': 'C++', 'c/c++': 'C++', 'modern c++': 'C++',
    'csharp': 'C#', 'c sharp': 'C#', 'c-sharp': 'C#',
    'objective c': 'Objective-C', 'objc': 'Objective-C', 'obj-c': 'Objective-C',
    'golang': 'Go', 'go lang': 'Go',
    'kotlin/jvm': 'Kotlin',
    'plsql': 'PL/SQL', 'pl/sql': 'PL/SQL', 'pl-sql': 'PL/SQL',
    'tsql': 'T-SQL',
    'shell': 'Bash', 'shell scripting': 'Bash', 'bash scripting': 'Bash',
    'html5': 'HTML', 'html 5': 'HTML',
    'css3': 'CSS', 'css 3': 'CSS',
    'sass': 'SCSS',
    # --- JS frameworks / libs ---
    'reactjs': 'React', 'react.js': 'React', 'react js': 'React',
    'vuejs': 'Vue', 'vue.js': 'Vue', 'vue js': 'Vue',
    'angularjs': 'Angular', 'angular.js': 'Angular', 'angular js': 'Angular', 'angular 2+': 'Angular',
    'nextjs': 'Next.js', 'next js': 'Next.js',
    'nuxtjs': 'Nuxt.js', 'nuxt js': 'Nuxt.js',
    'nodejs': 'Node.js', 'node': 'Node.js', 'node js': 'Node.js',
    'expressjs': 'Express', 'express.js': 'Express', 'express js': 'Express',
    'nestjs': 'Nest.js', 'nest js': 'Nest.js', 'nest.js': 'Nest.js',
    'redux toolkit': 'Redux', 'rtk': 'Redux',
    'tailwindcss': 'Tailwind', 'tailwind css': 'Tailwind',
    'material ui': 'Material-UI', 'mui': 'Material-UI',
    # --- Python ---
    'scikit learn': 'Scikit-learn', 'sklearn': 'Scikit-learn',
    'tensorflow 2': 'TensorFlow', 'tf2': 'TensorFlow', 'tf': 'TensorFlow',
    'pytorch lightning': 'PyTorch',
    'hugging face': 'HuggingFace', 'huggingface transformers': 'HuggingFace',
    'transformers': 'HuggingFace',
    'open ai': 'OpenAI', 'openai api': 'OpenAI', 'chatgpt': 'OpenAI',
    'gpt-4': 'OpenAI', 'gpt-3.5': 'OpenAI',
    'llms': 'LLM', 'large language model': 'LLM', 'large language models': 'LLM',
    'fast api': 'FastAPI',
    # --- Java / .NET ---
    'spring boot': 'Spring', 'spring framework': 'Spring', 'spring mvc': 'Spring',
    'spring data': 'Spring', 'spring cloud': 'Spring', 'spring security': 'Spring',
    'dotnet': '.NET', 'net core': '.NET', '.net core': '.NET',
    '.net 6': '.NET', '.net 7': '.NET', '.net 8': '.NET', '.net framework': '.NET',
    'aspnet': 'ASP.NET', 'asp net': 'ASP.NET', 'asp.net core': 'ASP.NET',
    'ef core': 'Entity Framework', 'entity framework core': 'Entity Framework',
    # --- Databases ---
    'postgres': 'PostgreSQL', 'postgressql': 'PostgreSQL', 'pgsql': 'PostgreSQL',
    'mongo': 'MongoDB', 'mongo db': 'MongoDB',
    'ms sql': 'MS SQL Server', 'sql server': 'MS SQL Server', 'mssql': 'MS SQL Server',
    't-sql': 'MS SQL Server',
    'oracle db': 'Oracle', 'oracle database': 'Oracle', 'oracle sql': 'Oracle',
    'elastic search': 'Elasticsearch', 'elastic': 'Elasticsearch',
    'big query': 'BigQuery', 'google bigquery': 'BigQuery',
    'dynamo db': 'DynamoDB', 'aws dynamodb': 'DynamoDB', 'amazon dynamodb': 'DynamoDB',
    'clickhouse db': 'ClickHouse',
    # --- Cloud providers ---
    'amazon web services': 'AWS', 'aws cloud': 'AWS',
    'google cloud': 'GCP', 'google cloud platform': 'GCP', 'gcloud': 'GCP',
    'microsoft azure': 'Azure', 'ms azure': 'Azure', 'azure cloud': 'Azure',
    'digital ocean': 'DigitalOcean',
    # --- AWS services (drop the AWS/Amazon prefix) ---
    'aws lambda': 'Lambda', 'amazon lambda': 'Lambda',
    'aws s3': 'S3', 'amazon s3': 'S3',
    'aws ec2': 'EC2', 'amazon ec2': 'EC2',
    'aws rds': 'RDS', 'amazon rds': 'RDS',
    'aws sqs': 'SQS', 'amazon sqs': 'SQS',
    'aws sns': 'SNS', 'amazon sns': 'SNS',
    'aws ecs': 'ECS', 'amazon ecs': 'ECS',
    'aws eks': 'EKS', 'amazon eks': 'EKS',
    'aws cloudformation': 'CloudFormation',
    'aws api gateway': 'API Gateway', 'amazon api gateway': 'API Gateway',
    'aws cloudwatch': 'CloudWatch', 'amazon cloudwatch': 'CloudWatch',
    'aws iam': 'IAM',
    'aws redshift': 'Redshift', 'amazon redshift': 'Redshift',
    # --- DevOps / Infra ---
    'k8s': 'Kubernetes', 'kuber': 'Kubernetes', 'kubernetes (k8s)': 'Kubernetes',
    'terraform iac': 'Terraform',
    'ci cd': 'CI/CD', 'cicd': 'CI/CD',
    'ci/cd pipelines': 'CI/CD', 'ci/cd pipeline': 'CI/CD',
    'continuous integration': 'CI/CD', 'continuous delivery': 'CI/CD',
    'continuous deployment': 'CI/CD',
    'gha': 'GitHub Actions', 'github action': 'GitHub Actions',
    'gitlab ci': 'GitLab CI', 'gitlab cicd': 'GitLab CI', 'gitlab ci/cd': 'GitLab CI',
    'gitlab pipelines': 'GitLab CI',
    'helm charts': 'Helm',
    'prometheus monitoring': 'Prometheus',
    'elk stack': 'ELK', 'kibana': 'ELK', 'logstash': 'ELK',
    'apache kafka': 'Kafka',
    'apache airflow': 'Airflow',
    'apache spark': 'Spark', 'pyspark': 'Spark',
    'docker compose': 'Docker',
    # --- ML / AI ---
    'ml': 'Machine Learning', 'machine-learning': 'Machine Learning',
    'dl': 'Deep Learning', 'deep-learning': 'Deep Learning',
    'nlp': 'NLP', 'natural language processing': 'NLP',
    'computer-vision': 'Computer Vision',
    'open cv': 'OpenCV', 'opencv-python': 'OpenCV',
    'ml ops': 'MLOps', 'ml-ops': 'MLOps',
    # --- APIs / Protocols ---
    'rest api': 'REST', 'rest apis': 'REST', 'restful': 'REST',
    'restful api': 'REST', 'restful apis': 'REST',
    'graph ql': 'GraphQL', 'graph-ql': 'GraphQL',
    'web socket': 'WebSocket', 'web sockets': 'WebSocket', 'websockets': 'WebSocket',
    'open api': 'OpenAPI', 'swagger / openapi': 'OpenAPI', 'swagger': 'OpenAPI',
    'protocol buffers': 'Protobuf',
    # --- Methodologies ---
    'agile / scrum': 'Agile', 'agile/scrum': 'Agile',
    'agile methodology': 'Agile', 'agile methodologies': 'Agile',
    'tdd': 'TDD', 'test driven development': 'TDD', 'test-driven development': 'TDD',
    'bdd': 'BDD', 'behavior driven development': 'BDD', 'behavior-driven development': 'BDD',
    'ddd': 'DDD', 'domain driven design': 'DDD', 'domain-driven design': 'DDD',
    'oop': 'OOP', 'object oriented programming': 'OOP', 'object-oriented programming': 'OOP',
    'fp': 'Functional Programming', 'functional-programming': 'Functional Programming',
    'micro services': 'Microservices', 'micro-services': 'Microservices',
    # --- BI / Analytics ---
    'power bi': 'Power BI', 'powerbi': 'Power BI', 'ms power bi': 'Power BI',
    'data studio': 'Looker Studio', 'google data studio': 'Looker Studio',
    'google analytics 4': 'Google Analytics', 'ga4': 'Google Analytics', 'ga': 'Google Analytics',
    'ms excel': 'Excel', 'microsoft excel': 'Excel',
    # --- Mobile ---
    'react-native': 'React Native', 'react native (rn)': 'React Native',
    'jetpack compose': 'Jetpack Compose',
    'swift ui': 'SwiftUI',
    # --- Misc ---
    'pytest framework': 'Pytest',
    'json api': 'JSON',
    'yml': 'YAML',
    'sso': 'SSO', 'single sign-on': 'SSO', 'single sign on': 'SSO',
    'tls': 'TLS/SSL', 'ssl': 'TLS/SSL', 'tls/ssl': 'TLS/SSL',
    'jwt tokens': 'JWT',
    'oauth2': 'OAuth', 'oauth 2': 'OAuth', 'oauth 2.0': 'OAuth',
    'a/b testing': 'A/B Testing',
    'meta ads': 'Facebook Ads',
}

# Case canonicalization: pick the canonical case when the lower-case form
# matches a popular skill that differs only by capitalization.
CASE_CANONICAL = {
    'pandas': 'Pandas', 'numpy': 'NumPy', 'scipy': 'SciPy',
    'scikit-learn': 'Scikit-learn',
    'pytest': 'Pytest', 'microservices': 'Microservices',
    'powerbi': 'Power BI', 'jira': 'Jira',
}

VERSION_SUFFIX = re.compile(r'\s+v?\d+(?:\.\d+)*\s*$')

def clean_skill(s: str):
    """Returns canonical skill name, or None if the term should be dropped."""
    s = s.strip()
    if not s:
        return None
    key = s.lower()
    if key in BLACKLIST:
        return None
    if key in NORMALIZATION_MAP:
        return NORMALIZATION_MAP[key]
    if key in CASE_CANONICAL:
        return CASE_CANONICAL[key]
    # Strip version suffix and retry
    stripped = VERSION_SUFFIX.sub('', s).strip()
    stripped_key = stripped.lower()
    if stripped_key != key:
        if stripped_key in BLACKLIST:
            return None
        if stripped_key in NORMALIZATION_MAP:
            return NORMALIZATION_MAP[stripped_key]
        if stripped_key in CASE_CANONICAL:
            return CASE_CANONICAL[stripped_key]
        return stripped
    return s

## 4. Apply Cleaning

In [8]:
def normalize_list(lst):
    out = {}
    for s in lst:
        cleaned = clean_skill(str(s))
        if cleaned:
            # Case-insensitive dedup within a single vacancy
            out[cleaned.lower()] = cleaned
    return sorted(out.values())

df_p['skills_raw']    = df_p['global_id'].map(raw_skills).apply(lambda x: x if isinstance(x, list) else [])
df_p['skills']        = df_p['skills_raw'].apply(normalize_list)
df_p['skills_count']  = df_p['skills'].str.len()
df_p = df_p.drop(columns=['skills_raw'])

df_p.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Rows:  {len(df_p):,}')

Saved: /kaggle/working/step3_vacancies_with_skills.parquet
Rows:  27,720


## 5. Summary & Diagnostics

Check the top-100 below for any remaining duplicates or noise. If you find any, extend `NORMALIZATION_MAP` / `BLACKLIST` and re-run sections 3-4.

In [9]:
print(f'With ≥1 skill:        {(df_p["skills_count"] >= 1).sum():,} ({(df_p["skills_count"] >= 1).mean()*100:.1f}%)')
print(f'Mean skills/vacancy:  {df_p["skills_count"].mean():.2f}')
print(f'Median:               {df_p["skills_count"].median():.0f}')

by_cat = df_p.groupby('category').agg(
    total=('skills_count', 'size'),
    with_skills=('skills_count', lambda x: (x >= 1).sum()),
).assign(pct=lambda d: (d['with_skills'] / d['total'] * 100).round(1))
by_cat = by_cat.sort_values('total', ascending=False).head(25)
print('\nCoverage by top 25 categories:')
print(by_cat.to_string())

With ≥1 skill:        19,797 (71.4%)
Mean skills/vacancy:  7.21
Median:               6

Coverage by top 25 categories:
                 total  with_skills   pct
category                                 
Marketing         3200         1291  40.3
Python            2249         2240  99.6
DevOps            2024         1992  98.4
Data Science      1921         1851  96.4
Sales             1424          295  20.7
Data Engineer     1114         1101  98.8
Project Manager    839          469  55.9
Support            828          392  47.3
Other              726          435  59.9
Design             682          153  22.4
Fullstack          615          613  99.7
HR                 599          175  29.2
Golang             552          548  99.3
QA                 531          504  94.9
Product Manager    477          322  67.5
SEO                474          197  41.6
Node.js            444          442  99.5
.NET               436          435  99.8
C++                391          341  87.

In [10]:
ctr = Counter()
for s in df_p['skills']:
    ctr.update(s)
print(f'Unique skills:        {len(ctr):,}')
print(f'Total skill mentions: {sum(ctr.values()):,}')

print('\nTop 100 skills (after normalization):')
for sk, cnt in ctr.most_common(100):
    print(f'  {sk:<32} {cnt:,}')

Unique skills:        34,780
Total skill mentions: 199,959

Top 100 skills (after normalization):
  Python                           7,268
  Docker                           4,268
  AWS                              4,178
  Kubernetes                       4,154
  PostgreSQL                       3,962
  SQL                              3,382
  Agile                            2,923
  Git                              2,744
  CI/CD                            2,543
  Scrum                            2,415
  REST                             1,828
  Azure                            1,791
  React                            1,699
  JavaScript                       1,689
  TypeScript                       1,649
  Terraform                        1,609
  GCP                              1,602
  Node.js                          1,438
  Redis                            1,406
  Linux                            1,326
  Jira                             1,261
  MySQL                            1,180
